# VeriCharge — revision steps (Colab)

These cells produce the artifacts used by the revision analyses in `analysis/`. They run **inference only** on the frozen models and do not modify any frozen result.

They assume the original pipeline (`01_VeriCharge_pipeline.ipynb`) has been run and its outputs saved under `MyDrive/VeriCharge/`. Use a GPU runtime for B1 and B2.

| Step | Purpose | Output |
|---|---|---|
| R-1 | Bundle frozen artifacts | `vericharge_revision_bundle.zip` |
| B1 | Five-seed deep-ensemble predictions | `results/revision_B1_ensemble_predictions.npz` |
| B2 | Telemetry-fault injection (6 faults) | `results/revision_B2_fault_injection.npz` |

After running, unzip the bundle into `artifacts/` and copy the two `.npz` files into `artifacts/results/`, then run the scripts in `analysis/` (see the repository README).

## R-1 — Bundle frozen artifacts

In [ ]:
# ============================================================
# VeriCharge — Revision Step R-1
# Bundle frozen artifacts for external analysis (read-only)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, json, numpy as np, pandas as pd

BASE = Path("/content/drive/MyDrive/VeriCharge")
OUT  = Path("/content/vericharge_revision_bundle")
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)

files = [
    "data/forecasting/caltech_forecasting_15m.npz",
    "data/forecasting/caltech_forecasting_metadata.csv.gz",
    "data/forecasting/caltech_forecasting_metadata_roles.csv.gz",
    "results/forecasting_dataset_config.json",
    "results/vericharge_frozen_deployment_policy.json",
    "results/vericharge_frozen_trust_thresholds.csv",
    "results/grid_operating_thresholds_training.csv",
    "results/trust_feature_architecture.json",
    "results/tcn_five_seed_modelvalidation_predictions.npz",
    "results/tcn_mc_uncertainty_modelvalidation.npz",
    "results/load_perturbation_robustness_modelvalidation.npz",
    "results/shift_score_modelvalidation.npz",
    "results/trust_features_modelvalidation.npz",
    "results/tcn_mc_uncertainty_trustcalibration.npz",
    "results/load_perturbation_robustness_trustcalibration.npz",
    "results/shift_score_trustcalibration.npz",
    "results/trust_features_trustcalibration.npz",
    "results/trustcalibration_failure_labels.npz",
    "results/trustcalibration_internal_split.npz",
    "results/trustcalibration_internal_split_config.json",
    "results/primary_trust_probabilities.npz",
    "results/finaltest_forecast_uncertainty.npz",
    "results/finaltest_robustness_shift.npz",
    "results/finaltest_trust_predictions.npz",
    "results/shift_detector_parameters.npz",
    "results/shift_score_training.npz",
    "results/trust_models/logistic_trust_15min.joblib",
    "results/trust_models/logistic_trust_30min.joblib",
    "results/trust_models/logistic_trust_60min.joblib",
    "results/trust_models/trust_scaler_15min.joblib",
    "results/trust_models/trust_scaler_30min.joblib",
    "results/trust_models/trust_scaler_60min.joblib",
]

missing = []
for rel in files:
    src = BASE / rel
    if src.exists():
        dst = OUT / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    else:
        missing.append(rel)

report = {}
meta = pd.read_csv(BASE / "data/forecasting/caltech_forecasting_metadata_roles.csv.gz", nrows=5)
report["metadata_roles_columns"] = list(meta.columns)
for p in OUT.rglob("*.npz"):
    z = np.load(p, allow_pickle=True)
    report[str(p.relative_to(OUT))] = {k: list(np.asarray(z[k]).shape) for k in z.files}
seed_dir = BASE / "models/tcn/seed_stability"
report["seed_models"] = sorted(x.name for x in seed_dir.glob("*")) if seed_dir.exists() else "NOT FOUND"
with open(OUT / "schema_report.json", "w") as f:
    json.dump(report, f, indent=2)

zip_path = shutil.make_archive("/content/vericharge_revision_bundle", "zip", OUT)
print("Missing files:", missing if missing else "none")
print("Seed models:", report["seed_models"])
print(f"Bundle: {zip_path}  ({Path(zip_path).stat().st_size/1024**2:.1f} MB)")


## B1 — Deep-ensemble predictions
Expected reproduction check: TEST seed 42 ≈ 0.002 kW; MV/DEV ≈ 0.04 kW (known small offset).

In [ ]:
# ============================================================
# VeriCharge — Revision Step B1
# Deep-ensemble predictions from the five frozen TCN seeds
# Inference only: NO training, NO frozen artifacts modified
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, time

BASE    = Path("/content/drive/MyDrive/VeriCharge")
DATA    = BASE / "data/forecasting"
RESULTS = BASE / "results"
MODELS  = BASE / "models/tcn/seed_stability"
SEEDS   = [1, 7, 21, 42, 100]

data = np.load(DATA / "caltech_forecasting_15m.npz")
meta = pd.read_csv(DATA / "caltech_forecasting_metadata_roles.csv.gz")
X_load   = data["X_load"].astype(np.float32)
X_active = data["X_active"].astype(np.float32)
Y = np.column_stack([data["y_15"], data["y_30"], data["y_60"]]).astype(np.float32)
roles = meta["development_role"].values

idx = {
    "train": np.where(roles == "ModelTraining")[0],
    "mv":    np.where(roles == "ModelValidation")[0],
    "dev":   np.where(roles == "TrustCalibration")[0],   # TrustFit + TrustCal (5094)
    "test":  np.where(roles == "FinalTest")[0],
}

# Training-only scaling (identical to the original pipeline)
tr = idx["train"]
load_mean, load_std = float(X_load[tr].mean(dtype=np.float64)), float(X_load[tr].std(dtype=np.float64))
act_mean,  act_std  = float(X_active[tr].mean(dtype=np.float64)), float(X_active[tr].std(dtype=np.float64))
y_mean = Y[tr].mean(axis=0, dtype=np.float64)
y_std  = Y[tr].std(axis=0, dtype=np.float64)

def scaled(ix):
    return np.stack([(X_load[ix]-load_mean)/load_std,
                     (X_active[ix]-act_mean)/act_std], axis=-1).astype(np.float32)

Xs = {k: scaled(v) for k, v in idx.items() if k != "train"}

preds = {}
t0 = time.time()
for s in SEEDS:
    # safe_mode=False is required because the TCN contains a Lambda layer (own trusted model files)
    model = tf.keras.models.load_model(MODELS / f"tcn_seed_{s}.keras", compile=False, safe_mode=False)
    for split in ["mv", "dev", "test"]:
        p = model.predict(Xs[split], batch_size=2048, verbose=0)
        preds[f"{split}_seed_{s}"] = (p * y_std + y_mean).astype(np.float32)
    del model; tf.keras.backend.clear_session()
    print(f"seed {s:>3} done  ({time.time()-t0:.1f}s)")

mv_frozen   = np.load(RESULTS / "tcn_five_seed_modelvalidation_predictions.npz")
test_frozen = np.load(RESULTS / "finaltest_forecast_uncertainty.npz")
dev_frozen  = np.load(RESULTS / "trustcalibration_failure_labels.npz")
assert np.array_equal(idx["mv"],   mv_frozen["sample_indices"])
assert np.array_equal(idx["test"], test_frozen["sample_indices"])
assert np.array_equal(idx["dev"],  dev_frozen["sample_indices"])

print("\nREPRODUCTION CHECK (mean |reconstructed - frozen| kW)")
for s in SEEDS:
    print(f"  MV   seed {s:>3}: {np.abs(preds[f'mv_seed_{s}'] - mv_frozen[f'seed_{s}']).mean():.4f}")
print(f"  DEV  seed  42: {np.abs(preds['dev_seed_42']  - dev_frozen['y_pred']).mean():.4f}")
print(f"  TEST seed  42: {np.abs(preds['test_seed_42'] - test_frozen['deterministic']).mean():.4f}")

out = RESULTS / "revision_B1_ensemble_predictions.npz"
np.savez_compressed(out, seeds=np.array(SEEDS),
                    mv_idx=idx["mv"], dev_idx=idx["dev"], test_idx=idx["test"], **preds)
print(f"\nSaved: {out}  ({out.stat().st_size/1024**2:.2f} MB)")


## B2 — Telemetry-fault injection
Expected checks on the clean scenario: yhat ≈ 0.002 kW, R < 0.01, S = 0.

In [ ]:
# ============================================================
# VeriCharge — Revision Step B2
# Telemetry-fault injection on FinalTest
# Recompute yhat, deep-ensemble E, MC-dropout U, robustness R, shift S
# Inference only: NO training, NO frozen artifacts modified
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, time

BASE    = Path("/content/drive/MyDrive/VeriCharge")
DATA    = BASE / "data/forecasting"
RESULTS = BASE / "results"
MODELS  = BASE / "models/tcn/seed_stability"
SEEDS   = [1, 7, 21, 42, 100]

data = np.load(DATA / "caltech_forecasting_15m.npz")
meta = pd.read_csv(DATA / "caltech_forecasting_metadata_roles.csv.gz")
X_load   = data["X_load"].astype(np.float32)
X_active = data["X_active"].astype(np.float32)
roles = meta["development_role"].values
train_idx = np.where(roles == "ModelTraining")[0]
test_idx  = np.where(roles == "FinalTest")[0]

frozen = np.load(RESULTS / "finaltest_forecast_uncertainty.npz")
frozen_rs = np.load(RESULTS / "finaltest_robustness_shift.npz")
assert np.array_equal(test_idx, frozen["sample_indices"])

# Frozen scaling (training-only inputs; frozen target inverse from the original Step 11A)
load_mean = float(X_load[train_idx].mean(dtype=np.float64)); load_std = float(X_load[train_idx].std(dtype=np.float64))
act_mean  = float(X_active[train_idx].mean(dtype=np.float64)); act_std  = float(X_active[train_idx].std(dtype=np.float64))
target_scale  = np.array([23.2071685902, 23.2072677468, 23.2073173366])
target_center = np.array([22.0984001175, 22.0928077608, 22.0775031814])
inv = lambda raw: (np.asarray(raw, np.float64) * target_scale + target_center).astype(np.float32)
def scale(L, A):
    return np.stack([(L - load_mean) / load_std, (A - act_mean) / act_std], axis=-1).astype(np.float32)

# Frozen shift detector (original Step 11B)
sp = np.load(RESULTS / "shift_detector_parameters.npz", allow_pickle=True)
def shift_features_10(L, A):
    L = np.asarray(L, np.float64); A = np.asarray(A, np.float64)
    t = np.arange(16, dtype=np.float64); tc = t - t.mean()
    last4 = L[:, -16:]; trend = ((last4 - last4.mean(1, keepdims=True)) @ tc) / np.sum(tc**2)
    d = np.abs(np.diff(L, axis=1))
    return np.column_stack([L[:, -1], L.mean(1), L.std(1), L[:, -16:].mean(1), trend,
                            d.mean(1), d.max(1), (L == 0).mean(1), A[:, -1], A.mean(1)])
def shift_score(F):
    z = (F - sp["scaler_center"]) / sp["scaler_scale"]; dlt = z - sp["location"]
    dist = np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", dlt, sp["precision"], dlt), 0))
    return np.searchsorted(sp["train_distance_sorted"], dist, side="right") / len(sp["train_distance_sorted"])

models = {s: tf.keras.models.load_model(MODELS / f"tcn_seed_{s}.keras", compile=False, safe_mode=False) for s in SEEDS}
primary = models[42]
det = lambda m, X: inv(m.predict(X, batch_size=4096, verbose=0))

def mc_dropout(X, passes=50, batch=4096):
    out = np.empty((passes, len(X), 3), np.float32)
    for p in range(passes):
        for st in range(0, len(X), batch):
            out[p, st:st+batch] = inv(primary(X[st:st+batch], training=True).numpy())
    return out.std(axis=0)            # numpy default ddof=0, identical to the original Step 11A

SEVERITIES = {"mild": 0.4110, "moderate": 0.8220, "strong": 1.6439}
def robustness(L, A, yhat):
    rng = np.random.default_rng(42); norm = []
    for sigma in SEVERITIES.values():
        acc = np.zeros_like(yhat)
        for _ in range(20):
            Lp = np.maximum(L + rng.normal(0, sigma, size=L.shape).astype(np.float32), 0)
            acc += np.abs(det(primary, scale(Lp, A)) - yhat)
        norm.append(acc / 20 / sigma)
    return np.mean(norm, axis=0).astype(np.float32)

L0, A0 = X_load[test_idx].copy(), X_active[test_idx].copy()
frng = np.random.default_rng(2026)
spike_pos = frng.integers(88, 96, size=len(L0))       # one step within the last 2 h

def fault(name):
    L, A = L0.copy(), A0.copy()
    if name == "clean":         pass
    elif name == "dropout_2h":  L[:, -8:] = 0.0
    elif name == "flatline_4h": L[:, -16:] = L[:, [-17]]
    elif name == "stale_1h":    L[:, -4:] = L[:, [-5]]
    elif name == "spike":       L[np.arange(len(L)), spike_pos] += 100.0
    elif name == "half_scale":  L *= 0.5
    elif name == "sessions_lost": A[:] = 0.0
    return L, A

SCEN = ["clean", "dropout_2h", "flatline_4h", "stale_1h", "spike", "half_scale", "sessions_lost"]
y_true = frozen["y_true"].astype(np.float32)
save, rows = {}, []
t0 = time.time()
for sc in SCEN:
    L, A = fault(sc); X = scale(L, A)
    yhat = det(primary, X)
    ens  = np.stack([det(models[s], X) for s in SEEDS])
    U    = mc_dropout(X)
    Rr   = robustness(L, A, yhat)
    F    = shift_features_10(L, A); S = shift_score(F)
    save.update({f"{sc}_yhat": yhat, f"{sc}_ens": ens, f"{sc}_U": U, f"{sc}_R": Rr,
                 f"{sc}_S": S, f"{sc}_shiftfeat": F.astype(np.float32)})
    mae = np.abs(y_true - yhat).mean(0)
    rows.append(dict(scenario=sc, MAE15=mae[0], MAE30=mae[1], MAE60=mae[2]))
    print(f"{sc:>14} done ({time.time()-t0:.0f}s)")
    if sc == "clean":
        print("  CHECK yhat vs frozen  :", np.abs(yhat - frozen["deterministic"]).mean().round(5), "kW")
        print("  CHECK R    vs frozen  :", np.abs(Rr - frozen_rs["combined_normalized_sensitivity"]).mean().round(5))
        print("  CHECK S    vs frozen  :", np.abs(S - frozen_rs["training_percentile"]).mean().round(6))

print(pd.DataFrame(rows).round(3).to_string(index=False))
out = RESULTS / "revision_B2_fault_injection.npz"
np.savez_compressed(out, scenarios=np.array(SCEN), test_idx=test_idx, spike_pos=spike_pos, **save)
print(f"Saved: {out} ({out.stat().st_size/1024**2:.1f} MB)")
